<a href="https://colab.research.google.com/github/dzakialthalsy/Machine_Learning_Dicoding/blob/main/Modul_3_Studi_Kasus_Hyperparameter_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Studi Kasus Hyperparameter Tuning

Bayangkan ketika kita bertugas untuk mengembangkan sebuah proyek machine learning. Kita bimbang kala memilih model yang akan dipakai dari 10 jenis model yang tersedia. Salah satu opsinya adalah dengan melatih semua model tersebut lalu membandingkan tingkat errornya pada test set. Setelah membandingkan performa semua model, kita mendapati model regresi linier memiliki tingkat error yang paling kecil katakanlah sebesar 5% dan membawa model tersebut ke tahap produksi.

Kemudian ketika model diuji pada tahap produksi, tingkat error ternyata sebesar 15%. Ini terjadi karena kita mengukur tingkat error berulang kali pada test set. Secara tidak sadar, kita telah memilih model yang hanya bekerja dengan baik pada test set tersebut. Hal ini menyebabkan model tidak bekerja dengan baik ketika menemui data baru. Solusi paling umum dari masalah ini adalah dengan melakukan hyperparameter tuning pada model machine learning.

Untuk memperdalam pengetahuan, pada materi ini kita akan melakukan dua percobaan hyperparameter tuning pada kasus regresi dan klasifikasi.

## Latihan Regression

### Preprocessing

kita akan menggunakan dataset fetch_california_housing dari Scikit-learn untuk melakukan regresi menggunakan Random Forest Regressor. dan tujuan model ini adalah memprediksi harga rata-rata rumah di wilayah tertentu.

Dalam studi kasus ini, kita akan melakukan hyperparameter tuning pada model Random Forest Regressor dan membandingkan performa serta efisiensi antara Grid Search, Random Search, dan Bayesian Optimization.

Pertama-tama, mari kita muat dataset yang akan digunakan. Pada latihan ini, kita tidak akan membahas pre-processing terlalu dalam dengan anggapan Anda sudah menguasai materi tersebut pada modul-modul sebelumnya.

In [2]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Muat dataset california housing
X, y = fetch_california_housing(return_X_y=True)

# Pisahkan data menjadi data latih dan data uji
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# melakukan scaling pada data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("shape of training data:", X_train.shape)
print("shape of testing data:", X_test.shape)

shape of training data: (14448, 8)
shape of testing data: (6192, 8)


### Training Model

latih model machine learning agar dapat memprediksi nilai kontinu karena kita akan membuat sebuah model regresi.

In [3]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

# Membuat model Random Forest Regressor
rf = RandomForestRegressor(random_state=42)
rf.fit(X_train, y_train)

# Evaluasi prediksi awal model tanpa tuning
y_pred = rf.predict(X_test)
# Menghitung RMSE
initial_mse = mean_squared_error(y_test, y_pred)
print("initial MSE on test set (without tuning):", initial_mse)

initial MSE on test set (without tuning): 0.25668110092783736


Nilai MSE pada model regresi ini 0.26 tanpa melakukan hyperparameter tuning. Nilai ini sudah cukup bagus mengingat kita menggunakan salah satu algoritma ensemble yaitu RandomForestRegressor.

In [4]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 10.1 MB/s eta 0:00:00


##### menggunakan model lainnya

In [5]:
!pip install catboost lightgbm

In [6]:
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor

# Membuat model KNN Regressor
knn = KNeighborsRegressor()
knn.fit(X_train, y_train)

# Evaluasi prediksi awal model KNN tanpa tuning
y_pred_knn = knn.predict(X_test)
initial_mse_knn = mean_squared_error(y_test, y_pred_knn)
print("Initial MSE on test set (KNN without tuning):", initial_mse_knn)

# Membuat model XGBoost Regressor
xgb = XGBRegressor(random_state=42)
xgb.fit(X_train, y_train)

# Evaluasi prediksi awal model XGBoost tanpa tuning
y_pred_xgb = xgb.predict(X_test)
initial_mse_xgb = mean_squared_error(y_test, y_pred_xgb)
print("Initial MSE on test set (XGBoost without tuning):", initial_mse_xgb)

# Membuat model CatBoost Regressor
catboost = CatBoostRegressor(random_state=42, verbose=0) # Set verbose to 0 to reduce output
catboost.fit(X_train, y_train)

# Evaluasi prediksi awal model CatBoost tanpa tuning
y_pred_catboost = catboost.predict(X_test)
initial_mse_catboost = mean_squared_error(y_test, y_pred_catboost)
print("Initial MSE on test set (CatBoost without tuning):", initial_mse_catboost)

# Membuat model LightGBM Regressor
lgbm = LGBMRegressor(random_state=42)
lgbm.fit(X_train, y_train)

# Evaluasi prediksi awal model LightGBM tanpa tuning
y_pred_lgbm = lgbm.predict(X_test)
initial_mse_lgbm = mean_squared_error(y_test, y_pred_lgbm)
print("Initial MSE on test set (LightGBM without tuning):", initial_mse_lgbm)

Initial MSE on test set (KNN without tuning): 0.42949402014873317
Initial MSE on test set (XGBoost without tuning): 0.21182273803266358
Initial MSE on test set (CatBoost without tuning): 0.19471021548230452
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001019 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1838
[LightGBM] [Info] Number of data points in the train set: 14448, number of used features: 8
[LightGBM] [Info] Start training from score 2.069240
Initial MSE on test set (LightGBM without tuning): 0.20910929437635892


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


kita dapat melihat bahwa model CatBoost ternyata memiliki mse score yang lebih rendah dibanding yang lain (lebih bagus)

### Hyperparameter Tuning

#### **hyperparameter tuning yang dimulai dengan grid search**

In [7]:
import time
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

start_time = time.time()  # Mencatat waktu mulai

# Definisikan parameter grid untuk Grid Search
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'criterion': ['squared_error', 'absolute_error'] # Corrected criterion values for regression
}

# Inisialisasi GridSearchCV
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=3, n_jobs=1, verbose=2)
grid_search.fit(X_train, y_train)

# Output hasil terbaik
print(f"Best parameters (Grid Search): {grid_search.best_params_}")
best_rf_grid = grid_search.best_estimator_

# Evaluasi performa model setelah Grid Search
y_pred_grid = best_rf_grid.predict(X_test)
grid_search_mse = mean_squared_error(y_test, y_pred_grid)
print(f"MSE after Grid Search: {grid_search_mse:.2f}")
end_time = time.time()  # mencatat waktu selesai
execution_time = end_time - start_time  # menghitung selisih waktu
print(f"Waktu eksekusi: {execution_time:.4f} detik")

Fitting 3 folds for each of 54 candidates, totalling 162 fits
[CV] END criterion=squared_error, max_depth=10, min_samples_split=2, n_estimators=100; total time=   9.9s


KeyboardInterrupt: 

In [ ]:
from sklearn.model_selection import GridSearchCV
import time
from sklearn.metrics import mean_squared_error
from catboost import CatBoostRegressor

start_time = time.time()

# definisikan parameter grid untuk grid search
param_grid = {
    'n_estimators': [100, 200, 500], # Using n_estimators as it's more common in sklearn
    'learning_rate' : [0.01, 0.05, 0.1],
    'depth' : [4, 6, 8],
    'l2_leaf_reg' : [1, 3, 5] # Corrected parameter name
}

# inisialisasi grid search
grid_search = GridSearchCV(estimator=catboost, param_grid=param_grid, cv=3, n_jobs=-1, verbose=2)
# latih model dengan grid search
grid_search.fit(X_train, y_train)

# output hasil terbaik
print("Best parameters found: ", grid_search.best_params_)
print("Best score found: ", grid_search.best_score_)
best_catboost_grid = grid_search.best_estimator_

# evaluasi performa model setelah grid search
y_pred_grid = best_catboost_grid.predict(X_test)
grid_search_mse= mean_squared_error(y_test, y_pred_grid)
print("MSE on test set after grid search:", grid_search_mse)

end_time = time.time() # mencatat waktu selesai
execution_time = end_time - start_time # menghitung selisih waktu
print("Execution time:", execution_time)

#### **Hyperparameter Tuning RandomSearch**

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint
import time
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

start_time_random = time.time()

# Definisikan parameter distribution untuk Randomized Search
param_dist = {
    'n_estimators': np.linspace(100, 500, 5, dtype=int),
    'max_depth': np.linspace(10, 50, 5, dtype=int),
    'min_samples_split': [2, 5, 10],
    'criterion': ['friedman_mse', 'squared_error']
}

# Inisialisasi RandomizedSearchCV
random_search = RandomizedSearchCV(estimator=rf, param_distributions=param_dist, n_iter=20, cv=3, n_jobs=1, verbose=2, random_state=42)

# Latih model dengan Randomized Search
random_search.fit(X_train, y_train)

# Output hasil terbaik
print(f"Best parameters (Random Search): {random_search.best_params_}")
best_rf_random = random_search.best_estimator_

# Evaluasi performa model setelah Randomized Search
y_pred_random = best_rf_random.predict(X_test)
random_search_mse = mean_squared_error(y_test, y_pred_random)
print(f"MSE after Random Search: {random_search_mse:.2f}")

end_time_random = time.time()
execution_time_random = end_time_random - start_time_random
print(f"Waktu eksekusi (Random Search): {execution_time_random:.4f} detik")

#### **Hyperparameter Tuning Bayesian Optimization**

In [10]:
!pip install scikit-optimize

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 5.0 MB/s eta 0:00:00


In [19]:
from skopt import BayesSearchCV
import time
from sklearn.metrics import mean_squared_error
from catboost import CatBoostRegressor

start_time = time.time() #mencatat waktu mulai

param_space = {
    'n_estimators': (100, 500),
    'max_depth': (10, 16),
}

# inisialisasi
bayes_search = BayesSearchCV(estimator=catboost, search_spaces=param_space, n_iter=32, cv=3, n_jobs=-1, verbose=2, random_state=42)
bayes_search.fit(X_train, y_train)

# output hasil terbaik
print("Best parameters found: ", bayes_search.best_params_)
best_catboost_bayes = bayes_search.best_estimator_

# evaluasi performa model setelah random search
y_pred_bayes = best_catboost_bayes.predict(X_test)
bayes_mse = mean_squared_error(y_test, y_pred_bayes)
print("MSE on test set after bayes search:", bayes_mse)
end_time = time.time() # mencatat waktu selesai
execution_time = end_time - start_time # menghitung selisih waktu
print("Execution time:", execution_time)

Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits


KeyboardInterrupt: 

In [21]:
from skopt import BayesSearchCV
import time
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor # Import RandomForestRegressor

start_time = time.time()  # Mencatat waktu mulai

# Definisikan ruang pencarian untuk Bayesian Optimization
param_space = {
    'n_estimators': (100, 500),
    'max_depth': (10, 50),
    'min_samples_split': (2, 10),
    'min_samples_leaf': (1, 4),
    'bootstrap': [True, False]
}

# Inisialisasi BayesSearchCV
bayes_search = BayesSearchCV(estimator=rf, search_spaces=param_space, n_iter=10, cv=3, n_jobs=-1, verbose=2, random_state=42) # Reduced n_iter
bayes_search.fit(X_train, y_train)

# Output hasil terbaik
print(f"Best parameters (Bayesian Optimization): {bayes_search.best_params_}")
best_rf_bayes = bayes_search.best_estimator_

# Evaluasi performa model setelah Random Search
y_pred_bayes = best_rf_bayes.predict(X_test)
bayes_mse = mean_squared_error(y_test, y_pred_bayes)
print(f"MSE after Grid Search: {bayes_mse:.2f}")
end_time = time.time()  # Mencatat waktu selesai
execution_time = end_time - start_time  # Menghitung selisih waktu
print(f"Waktu eksekusi: {execution_time:.4f} detik")

Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Best parameters (Bayesian Optimization): OrderedDict({'bootstrap': True, 'max_depth': 47, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 175})
MSE after Grid Search: 0.25
Waktu eksekusi: 938.8854 detik
